# Prompt Engineering Techniques

Notebook pratico che accompagna il modulo **05 - Prompt Engineering Techniques** del corso *Building with the Claude API*.

Copriamo, con esempi eseguibili, le tecniche ufficiali di prompt engineering di Anthropic:

1. Be clear and direct
2. Multishot prompting (esempi)
3. Let Claude think (Chain of Thought)
4. Use XML tags
5. Give Claude a role (system prompt)
6. Prefill Claude's response
7. Chain complex prompts
8. Long context tips

Per ogni tecnica: una breve spiegazione, un esempio "prima/dopo" quando ha senso, e una cella per sperimentare.


## Setup

Stessa struttura di base usata negli altri notebook del corso: client Anthropic, helper per costruire la conversation history, funzione `chat`.

In [1]:
# Import delle librerie necessarie
import json
from textwrap import dedent
from dotenv import load_dotenv
from anthropic import Anthropic

# Carica le variabili d'ambiente dal file .env (contiene ANTHROPIC_API_KEY)
load_dotenv()

# Inizializza il client Anthropic
client = Anthropic()

# Modello da usare: haiku è veloce ed economico, ottimo per esercitarsi
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


## 1. Be Clear and Direct

Claude non legge nel pensiero: risponde meglio quando l'istruzione è esplicita su **cosa** vuoi, **per chi** è l'output, **che formato** deve avere, e cosa evitare.

Regola pratica: se un collega umano nuovo del team avrebbe bisogno di ulteriori chiarimenti leggendo il tuo prompt, probabilmente anche Claude ne ha bisogno.

Confrontiamo un prompt vago con uno chiaro e diretto sullo stesso task.

In [2]:
# Prompt VAGO: mancano dettagli su lunghezza, tono, pubblico, formato
vague_prompt = "Scrivi qualcosa sul nostro nuovo prodotto: un'app di meditazione."

messages = []
add_user_message(messages, vague_prompt)
print("=== PROMPT VAGO ===")
print(chat(messages))


=== PROMPT VAGO ===


# Benvenuto nel tuo spazio di serenità

Presentiamo la nostra nuova **app di meditazione**: uno strumento progettato per aiutarti a ritrovare calma e consapevolezza nella frenesia quotidiana.

## Cosa offriamo

✨ **Meditazioni guidate** - Sessioni da 5 a 30 minuti, adatte a principianti e praticanti esperti

🎵 **Suoni rilassanti** - Ambienti sonori naturali per creare l'atmosfera perfetta

📅 **Programmi personalizzati** - Percorsi specifici per stress, sonno, focus e consapevolezza

🏆 **Tracciamento progressi** - Monitora il tuo cammino verso il benessere

## Perché sceglierci

- Interfaccia intuitiva e accogliente
- Comunità di meditanti per supporto e condivisione
- Disponibile su iOS e Android
- Primi 7 giorni completamente gratuiti

**La meditazione non è una fuga dalla realtà, ma un ritorno a se stessi.**

Scarica l'app oggi e inizia il tuo percorso verso la consapevolezza.

---

Vuoi che approfondisca un aspetto specifico o che adatti il tono per un pubblico particolare?


In [3]:
# Prompt CHIARO E DIRETTO: task, pubblico, tono, formato e vincoli espliciti
clear_prompt = dedent("""
    Scrivi una descrizione prodotto per una nuova app di meditazione chiamata "Calmly".

    Dettagli:
    - Pubblico: professionisti impegnati che non hanno mai meditato prima
    - Tono: rassicurante ma non new-age, concreto
    - Lunghezza: massimo 80 parole
    - Deve includere: una funzionalità distintiva (sessioni da 3 minuti), una call to action

    Non usare gergo da wellness (es. "energia", "vibrazioni", "risveglio interiore").
""").strip()

messages = []
add_user_message(messages, clear_prompt)
print("=== PROMPT CHIARO E DIRETTO ===")
print(chat(messages))


=== PROMPT CHIARO E DIRETTO ===


# Calmly - Meditazione per chi non ha tempo

Riduci lo stress in soli 3 minuti. Calmly è pensata per professionisti occupati: sessioni brevi, basate su metodi scientifici comprovati, senza complicazioni. Meditazione pratica che si adatta alla tua giornata.

Inizia oggi gratuitamente. Scopri come 180 secondi possono fare la differenza.

---

*(78 parole)*


## 2. Multishot Prompting (esempi)

Mostrare 2-5 esempi di input → output desiderato è spesso la tecnica con il rapporto sforzo/risultato più alto: Claude generalizza formato, tono e livello di dettaglio dagli esempi.

Gli esempi vanno racchiusi in tag XML (`<example>`) per separarli chiaramente dalle istruzioni.

Esempio: classificare il sentiment di recensioni con un formato di output molto specifico.

In [4]:
zero_shot_prompt = dedent("""
    Classifica il sentiment di questa recensione:

    "La spedizione ha impiegato due settimane ma il prodotto funziona bene."
""").strip()

messages = []
add_user_message(messages, zero_shot_prompt)
print("=== ZERO-SHOT (nessun esempio) ===")
print(chat(messages))


=== ZERO-SHOT (nessun esempio) ===


# Classificazione Sentiment

**Sentiment: MISTO (Neutro-Positivo)**

## Analisi:

**Aspetti Negativi:**
- Spedizione lenta (due settimane)

**Aspetti Positivi:**
- Prodotto funziona bene

## Valutazione complessiva:
La recensione è **prevalentemente positiva** con una nota critica. Il cliente è sostanzialmente soddisfatto del prodotto, ma esprime un'insoddisfazione sulla velocità della consegna. Nonostante il problema logistico, il tono rimane equilibrato e non emotivamente negativo.

**Score sentiment: 6.5/10** (positivo, ma con margini di miglioramento)


In [5]:
multishot_prompt = dedent("""
    Classifica il sentiment delle recensioni nel formato esatto mostrato negli esempi.

    <example>
    <review>Prodotto arrivato rotto, servizio clienti inutile.</review>
    <output>{"sentiment": "negative", "confidence": "high"}</output>
    </example>

    <example>
    <review>Fa il suo dovere, niente di eccezionale ma nessun problema.</review>
    <output>{"sentiment": "neutral", "confidence": "medium"}</output>
    </example>

    <example>
    <review>Superato ogni aspettativa, lo ricomprerei subito!</review>
    <output>{"sentiment": "positive", "confidence": "high"}</output>
    </example>

    Ora classifica questa recensione:
    <review>La spedizione ha impiegato due settimane ma il prodotto funziona bene.</review>
""").strip()

messages = []
add_user_message(messages, multishot_prompt)
print("=== MULTISHOT (con esempi) ===")
print(chat(messages))


=== MULTISHOT (con esempi) ===


```json
{"sentiment": "neutral", "confidence": "medium"}
```


## 3. Let Claude Think (Chain of Thought)

Per task che richiedono ragionamento (logica, matematica, decisioni con più criteri), far "pensare ad alta voce" Claude prima di dare la risposta finale migliora l'accuratezza.

Pattern consigliato: chiedere il ragionamento dentro `<thinking>` e la risposta finale dentro `<answer>`, così puoi facilmente estrarre solo la parte che ti serve senza mostrare il ragionamento all'utente finale.

In [6]:
problem = dedent("""
    Un negozio vende magliette a 18€ l'una. Se compri 3 o più magliette hai il 20% di sconto
    sul totale. Marco compra 4 magliette e usa anche un buono sconto da 5€.
    Quanto paga in totale?
""").strip()

# SENZA chain of thought: risposta diretta, più a rischio di errori di calcolo
messages = []
add_user_message(messages, problem)
print("=== SENZA CoT ===")
print(chat(messages, temperature=0))


=== SENZA CoT ===


# Calcolo del prezzo totale

**Passo 1: Prezzo senza sconti**
- 4 magliette × 18€ = 72€

**Passo 2: Applicare lo sconto del 20%**
- Marco compra 4 magliette, quindi ha diritto allo sconto
- Sconto: 72€ × 20% = 72€ × 0,20 = 14,40€
- Prezzo dopo sconto: 72€ - 14,40€ = **57,60€**

**Passo 3: Applicare il buono sconto da 5€**
- 57,60€ - 5€ = **52,60€**

**Marco paga in totale 52,60€**


In [7]:
cot_prompt = dedent("""
    {problem}

    Prima di rispondere, ragiona passo passo dentro i tag <thinking>.
    Poi scrivi la risposta finale, solo il numero in euro, dentro i tag <answer>.
""").strip().format(problem=problem)

messages = []
add_user_message(messages, cot_prompt)
result = chat(messages, temperature=0)
print("=== CON CoT ===")
print(result)


=== CON CoT ===
<thinking>
Passo 1: Calcolare il prezzo totale senza sconti
Marco compra 4 magliette a 18€ l'una
4 × 18€ = 72€

Passo 2: Applicare lo sconto del 20% (poiché compra 3 o più magliette)
Sconto del 20% su 72€
20% di 72€ = 0,20 × 72€ = 14,40€
Prezzo dopo sconto percentuale = 72€ - 14,40€ = 57,60€

Passo 3: Applicare il buono sconto da 5€
Prezzo finale = 57,60€ - 5€ = 52,60€
</thinking>

<answer>52,60</answer>


In [8]:
# Estrarre solo l'<answer> dal risultato, ignorando il <thinking>
import re

match = re.search(r"<answer>(.*?)</answer>", result, re.DOTALL)
if match:
    print("Risposta estratta:", match.group(1).strip())


Risposta estratta: 52,60


## 4. Use XML Tags

Claude è addestrato a riconoscere molto bene la struttura XML. Usare tag come `<document>`, `<instructions>`, `<data>`, `<example>` per separare le sezioni del prompt:

- riduce l'ambiguità su cosa sia istruzione e cosa sia contenuto
- permette di fare riferimento a sezioni specifiche nelle istruzioni stesse
- rende il prompt più facile da mantenere e modificare

Esempio: chiedere di confrontare due documenti, dove senza tag Claude potrebbe confondere dove finisce un testo e inizia l'altro.

In [9]:
xml_prompt = dedent("""
    Confronta le due policy di reso qui sotto e indica in una frase la differenza principale.

    <policy name="Negozio A">
    Reso gratuito entro 30 giorni, rimborso completo, nessuna domanda.
    </policy>

    <policy name="Negozio B">
    Reso entro 14 giorni con spese di spedizione a carico del cliente, solo buono acquisto.
    </policy>

    Rispondi facendo riferimento ai due negozi per nome.
""").strip()

messages = []
add_user_message(messages, xml_prompt)
print(chat(messages, temperature=0))


La differenza principale è che il Negozio A offre un reso gratuito con rimborso completo entro 30 giorni, mentre il Negozio B richiede il pagamento della spedizione e concede solo un buono acquisto entro 14 giorni.


## 5. Give Claude a Role (System Prompt)

Il parametro `system` non serve solo a dare "personalità": definisce il framing dell'intero task e cambia concretamente cosa Claude nota, quanto è tecnico, e cosa considera rilevante.

Confrontiamo la stessa domanda con due `system` prompt diversi.

In [10]:
code_snippet = dedent("""
    def get_user(id):
        query = "SELECT * FROM users WHERE id = " + id
        return db.execute(query)
""").strip()

question = f"Rivedi questo codice:\n\n{code_snippet}"

# Senza ruolo specifico
messages = []
add_user_message(messages, question)
print("=== SENZA RUOLO ===")
print(chat(messages, temperature=0))


=== SENZA RUOLO ===


# Revisione del codice

Questo codice ha **gravi problemi di sicurezza e qualità**. Ecco i principali:

## 🔴 Problemi

1. **SQL Injection** - Vulnerabilità critica di sicurezza
2. **Mancanza di validazione** dell'input
3. **Nessuna gestione degli errori**
4. **Type hint mancanti**
5. **Docstring assente**

## ✅ Codice corretto

```python
def get_user(user_id: int) -> dict | None:
    """
    Recupera un utente dal database per ID.
    
    Args:
        user_id: ID dell'utente
        
    Returns:
        Dizionario con i dati dell'utente o None se non trovato
        
    Raises:
        ValueError: Se l'ID non è valido
    """
    # Validazione input
    if not isinstance(user_id, int) or user_id <= 0:
        raise ValueError("L'ID deve essere un numero positivo")
    
    try:
        # Usa parametri preparati (query parameterized)
        query = "SELECT * FROM users WHERE id = ?"
        result = db.execute(query, (user_id,))
        return result.fetchone()
    
    except Exce

In [11]:
security_reviewer_system = (
    "Sei un revisore di codice senior specializzato in sicurezza applicativa. "
    "Per ogni frammento di codice che ricevi, identifichi SOLO le vulnerabilità di sicurezza, "
    "citando la CWE se applicabile, e proponi la correzione minima necessaria."
)

messages = []
add_user_message(messages, question)
print("=== CON RUOLO 'security reviewer' ===")
print(chat(messages, system=security_reviewer_system, temperature=0))


=== CON RUOLO 'security reviewer' ===


# Revisione di Sicurezza

## Vulnerabilità Identificata

**SQL Injection** - CWE-89

### Problema
La concatenazione diretta della variabile `id` nella query SQL consente a un attaccante di iniettare codice SQL arbitrario.

**Esempio di exploit:**
```python
get_user("1 OR 1=1")  # Restituisce tutti gli utenti
get_user("1; DROP TABLE users;--")  # Elimina la tabella
```

---

## Correzione Minima

```python
def get_user(id):
    query = "SELECT * FROM users WHERE id = ?"
    return db.execute(query, (id,))
```

### Cosa cambia
- Utilizzo di **parametrized queries** (placeholder `?`)
- Passaggio del parametro come argomento separato
- Il driver del database gestisce automaticamente l'escape

### Note
- La sintassi esatta dipende dal driver DB (SQLAlchemy, psycopg2, sqlite3, ecc.)
- Esempio con SQLAlchemy: `db.execute(text("SELECT * FROM users WHERE id = :id"), {"id": id})`
- Aggiungere validazione del tipo: `id` dovrebbe essere un intero


## 6. Prefill Claude's Response

Puoi "iniziare tu" il messaggio dell'assistant (es. con `{` o ```` ```json ````) per forzare il formato di output e saltare preamboli tipo "Certo, ecco...".

È la stessa tecnica già usata nel notebook di Prompt Evaluation di questo corso (`add_assistant_message(messages, "```json")`).

In [12]:
extract_prompt = dedent("""
    Estrai nome, età e città da questo testo. Rispondi solo con JSON.

    "Mi chiamo Sara, ho 29 anni e vivo a Torino."
""").strip()

# SENZA prefill: Claude potrebbe aggiungere testo prima/dopo il JSON
messages = []
add_user_message(messages, extract_prompt)
print("=== SENZA PREFILL ===")
print(chat(messages, temperature=0))


=== SENZA PREFILL ===


```json
{
  "nome": "Sara",
  "età": 29,
  "città": "Torino"
}
```


In [13]:
# CON prefill: forziamo l'inizio della risposta, Claude continua da lì
messages = []
add_user_message(messages, extract_prompt)
add_assistant_message(messages, "{")

result = chat(messages, temperature=0, stop_sequences=["}"])
full_json = "{" + result + "}"

print("=== CON PREFILL ===")
print(full_json)
print(json.loads(full_json))


=== CON PREFILL ===
{
  "nome": "Sara",
  "età": 29,
  "città": "Torino"
}
{'nome': 'Sara', 'età': 29, 'città': 'Torino'}


## 7. Chain Complex Prompts

Per task complessi, invece di un unico mega-prompt, si spezza il lavoro in più chiamate: l'output della prima diventa input della seconda. Vantaggi:

- ogni singolo step è più semplice e più affidabile
- puoi ispezionare/validare i risultati intermedi
- puoi usare modelli diversi per step diversi (es. haiku per l'estrazione, un modello più potente per la sintesi)

Esempio: estrarre i punti chiave da un testo (step 1), poi trasformarli in un'email per un cliente (step 2).

In [14]:
meeting_notes = dedent("""
    Riunione con il cliente Acme Corp - 12 settembre.
    Il cliente lamenta lentezza nella dashboard durante le ore di punta.
    Vogliono l'export in PDF entro fine mese.
    Sono soddisfatti del supporto ricevuto finora.
    Budget per il prossimo trimestre non ancora confermato.
""").strip()

# STEP 1: estrarre solo i punti chiave, come lista
step1_prompt = f"""Estrai i punti chiave da queste note di riunione, come elenco puntato conciso:

<notes>
{meeting_notes}
</notes>"""

messages = []
add_user_message(messages, step1_prompt)
key_points = chat(messages, temperature=0)
print("=== STEP 1: punti chiave ===")
print(key_points)


=== STEP 1: punti chiave ===
# Punti Chiave - Riunione Acme Corp (12 settembre)

• **Problema tecnico**: Dashboard lenta durante le ore di punta
• **Richiesta urgente**: Export in PDF entro fine mese
• **Feedback positivo**: Soddisfatti del supporto ricevuto
• **Questione aperta**: Budget Q4 non ancora confermato


In [15]:
# STEP 2: usare l'output dello step 1 come input per generare l'email
step2_prompt = f"""Scrivi una breve email al cliente Acme Corp che riassume questi punti
in tono professionale e rassicurante, senza inventare informazioni non presenti:

<key_points>
{key_points}
</key_points>"""

messages = []
add_user_message(messages, step2_prompt)
email = chat(messages, temperature=0.5)
print("=== STEP 2: email finale ===")
print(email)


=== STEP 2: email finale ===
**Oggetto: Riepilogo riunione del 12 settembre – Acme Corp**

---

Caro [Nome Cliente],

Le scrivo per riepilogare i punti principali discussi durante il nostro incontro di mercoledì 12 settembre.

**Problematiche riscontrate**
Abbiamo preso nota del rallentamento della dashboard durante le ore di punta e stiamo già valutando soluzioni per ottimizzare le prestazioni.

**Sviluppi in corso**
Rimaniamo impegnati a implementare la funzionalità di export in PDF entro la fine del mese, come da vostra richiesta.

**Apprezzamenti**
Siamo molto soddisfatti di aver ricevuto il vostro feedback positivo riguardo alla qualità del supporto fornito. Continueremo a mantenerlo a questo livello.

**Prossimi passi**
Rimane in sospeso la conferma del budget per il Q4. Restiamo a disposizione per discuterne nei dettagli quando avrete chiarezza in merito.

Resto a vostra completa disposizione per qualsiasi chiarimento o necessità.

Cordiali saluti,

[Tuo Nome]


## 8. Long Context Tips

Quando il prompt contiene documenti lunghi:

- metti i documenti **prima** delle istruzioni/domanda, non dopo (Claude li elabora meglio se li legge per primi e poi sa cosa gli viene chiesto)
- avvolgili in tag XML con un identificatore (`<document index="1">`)
- per task di Q&A/analisi, chiedi a Claude di citare le frasi rilevanti dei documenti **prima** di rispondere: riduce le allucinazioni e rende la risposta verificabile

Esempio con due "documenti" corti (nella pratica potrebbero essere migliaia di parole ciascuno).

In [16]:
doc1 = dedent("""
    Report Q1: Le vendite sono cresciute del 12% rispetto al trimestre precedente,
    trainate principalmente dal mercato europeo. Il margine operativo e' sceso al 18%
    a causa dell'aumento dei costi logistici.
""").strip()

doc2 = dedent("""
    Report Q2: Le vendite sono cresciute ulteriormente del 5%, ma il margine operativo
    e' risalito al 22% grazie alla rinegoziazione dei contratti di spedizione.
""").strip()

long_context_prompt = dedent("""
    <document index="1" name="Report Q1">
    {doc1}
    </document>

    <document index="2" name="Report Q2">
    {doc2}
    </document>

    In base ai documenti sopra, il margine operativo e' migliorato o peggiorato dal Q1 al Q2?
    Prima cita tra virgolette le frasi rilevanti di ciascun documento dentro <quotes>,
    poi rispondi dentro <answer>.
""").strip().format(doc1=doc1, doc2=doc2)

messages = []
add_user_message(messages, long_context_prompt)
print(chat(messages, temperature=0))


<quotes>
Q1: "Il margine operativo e' sceso al 18% a causa dell'aumento dei costi logistici."

Q2: "il margine operativo e' risalito al 22% grazie alla rinegoziazione dei contratti di spedizione."
</quotes>

<answer>
Il margine operativo è migliorato dal Q1 al Q2. È aumentato dal 18% al 22%, un miglioramento di 4 punti percentuali, grazie alla rinegoziazione dei contratti di spedizione che ha permesso di ridurre i costi logistici.
</answer>


## Prova tu

Scegli una delle tecniche sopra e applicala a un tuo caso d'uso reale (es. un prompt che usi già in `game-portfolio` o in un altro modulo del corso). Prova prima senza la tecnica, poi con, e confronta i risultati.

In [17]:
# Spazio libero per i tuoi esperimenti
messages = []
add_user_message(messages, "")
# print(chat(messages))
